## Cell 1: Read Typing Committee proposals into source map

In [1]:
import json
import re
import time
import pandas as pd
import requests

# Load configuration
with open("config.json") as f:
    config = json.load(f)

# Get GitHub Token if available
GITHUB_TOKEN = config.get("github_token")

HEADERS = {
    "Accept": "application/vnd.github.v3+json",
    "User-Agent": "TypingCouncilSubmissionsScraper/1.0",
}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"token {GITHUB_TOKEN}"


def extract_pep_number(text):
    """Extract PEP number from title or body text (e.g., 'PEP 742' -> 742)."""
    match = re.search(r"\bPEP\s*(\d+)\b", str(text), re.IGNORECASE)
    return int(match.group(1)) if match else None


def extract_links(text):
    """Extract GitHub PR links and discuss.python.org discussion links from markdown text."""
    if not text:
        return None, None

    pr_matches = re.findall(
        r"https?://github\.com/[^/\s]+/[^/\s]+/pull/\d+", text
    )
    pr_link = pr_matches[0] if pr_matches else None

    disc_matches = re.findall(
        r"https?://discuss\.python\.org/t/[^\s\)\>\]\"']+", text
    )
    discussion_link = disc_matches[0] if disc_matches else None

    return pr_link, discussion_link


def fetch_typing_council_issues():
    """Fetch all issues (excluding PRs) from python/typing-council repository."""
    base_url = "https://api.github.com/repos/python/typing-council/issues"
    params = {"state": "all", "per_page": 100}

    all_issues = []
    page = 1

    print("Fetching submissions from python/typing-council...")
    while True:
        params["page"] = page
        response = requests.get(base_url, headers=HEADERS, params=params)

        if response.status_code != 200:
            print(f"Error fetching page {page} (Status: {response.status_code})")
            break

        data = response.json()
        if not data:
            break

        for item in data:
            if "pull_request" not in item:
                all_issues.append(item)

        if len(data) < params["per_page"]:
            break

        page += 1
        time.sleep(0.2)

    # Sort issues chronologically by Issue Number (Ascending)
    all_issues = sorted(all_issues, key=lambda x: x["number"])

    rows = []
    for seq_num, issue in enumerate(all_issues, start=1):
        issue_num = issue["number"]
        title = issue["title"]
        created_date = issue["created_at"][:10]
        vote_link = issue["html_url"]
        body = issue.get("body", "") or ""

        pep_num = extract_pep_number(title) or extract_pep_number(body)
        pr_link, discussion_link = extract_links(body)

        if (not pr_link or not discussion_link) and issue.get("comments", 0) > 0:
            comments_url = issue.get("comments_url")
            c_resp = requests.get(comments_url, headers=HEADERS)
            if c_resp.status_code == 200:
                for comment in c_resp.json():
                    c_body = comment.get("body", "") or ""
                    c_pr, c_disc = extract_links(c_body)
                    if not pr_link and c_pr:
                        pr_link = c_pr
                    if not discussion_link and c_disc:
                        discussion_link = c_disc

        rows.append(
            {
                "Seq #": seq_num,
                "Issue #": issue_num,
                "Date": created_date,
                "Topic / Proposal Title": title,
                "PEP #": pep_num if pep_num else "N/A",
                "Vote Link": vote_link,
                "Draft / Text Link (PR)": pr_link
                if pr_link
                else "N/A (No PR linked)",
                "Discussion Link": discussion_link
                if discussion_link
                else "N/A (No Discussion linked)",
            }
        )

    df_tc = pd.DataFrame(rows)
    print(f"Extracted {len(df_tc)} Typing Council submission issues.")
    return df_tc


df_typing_council = fetch_typing_council_issues()

Fetching submissions from python/typing-council...
Extracted 58 Typing Council submission issues.


## Cell 2: Read PEPs with polls into source map and Multi-PEP Explosion & Merging
This cell reads the Excel PEP source map, explodes any multi-PEP entries (e.g., 8001-8016 or 634, 635, 636, 640, 642) into individual rows per PEP, merges them with df_typing_council, re-indexes Seq #, and exports the dataset locally.

In [2]:
import os
import json
import re
import pandas as pd


def parse_pep_numbers(pep_val):
    """Parses PEP numbers into a list of strings, handling individual numbers,

    comma-separated values ('634, 635'), and ranges ('8001-8016').
    """
    if pd.isna(pep_val) or str(pep_val).strip().lower() in [
        "nan",
        "none",
        "",
        "n/a",
    ]:
        return ["N/A"]

    pep_str = str(pep_val).strip()

    # Split by comma or semicolon first
    raw_parts = re.split(r"[,;]", pep_str)
    peps = []

    for part in raw_parts:
        part = part.strip()
        # Check if part is a range like "8001-8016"
        range_match = re.match(r"^(\d+)\s*-\s*(\d+)$", part)
        if range_match:
            start, end = int(range_match.group(1)), int(range_match.group(2))
            peps.extend([str(i) for i in range(start, end + 1)])
        elif part:
            peps.append(part)

    return peps if peps else ["N/A"]


# 1. Read the PEP Vote Source Map Excel File
excel_filename = "python_vote_source_map-4.xlsx"
if not os.path.exists(excel_filename):
    excel_filename = "python_vote_source_map-4_2.xlsx"

df_excel = pd.read_excel(excel_filename)

# Filter out entries where Decision Category is 'Other' (keep PEPs)
pep_df = df_excel[
    df_excel["Decision Category"].astype(str).str.strip().str.upper() == "PEP"
].copy()

# 2. Map & Explode multi-PEP entries into individual rows
mapped_pep_rows = []
for _, row in pep_df.iterrows():
    source_url = (
        str(row["Source URL"]).strip() if pd.notna(row["Source URL"]) else "N/A"
    )

    # Derive PR link vs Discussion link from Source URL structure
    pr_link = (
        source_url
        if "github.com/" in source_url and "/pull/" in source_url
        else "N/A (No PR linked)"
    )
    disc_link = (
        source_url
        if "discuss.python.org" in source_url
        else "N/A (No Discussion linked)"
    )

    # Explode multi-PEP entries
    pep_val = row["PEP #"]
    parsed_peps = parse_pep_numbers(pep_val)

    for pep_item in parsed_peps:
        mapped_pep_rows.append(
            {
                "Issue #": "N/A",
                "Date": str(row["Date"])[:10],
                "Topic / Proposal Title": str(row["Decision Title"]),
                "PEP #": pep_item,
                "Vote Link": source_url,
                "Draft / Text Link (PR)": pr_link,
                "Discussion Link": disc_link,
            }
        )

df_pep_mapped = pd.DataFrame(mapped_pep_rows)

# 3. Merge Typing Council dataset with mapped & exploded Excel PEP dataset
df_combined = pd.concat([df_typing_council, df_pep_mapped], ignore_index=True)

# Re-assign continuous sequential numbers across all exploded rows
df_combined["Seq #"] = range(1, len(df_combined) + 1)

# Enforce uniform column ordering
columns_order = [
    "Seq #",
    "Issue #",
    "Date",
    "Topic / Proposal Title",
    "PEP #",
    "Vote Link",
    "Draft / Text Link (PR)",
    "Discussion Link",
]
df_combined = df_combined[columns_order]

## Cell 4: Upsert into MongoDB (consensus.python_decision_eps)
This cell connects to MongoDB using config["mongo_uri"], targets the consensus database and the python_decision_eps collection, and performs upserts using a composite filter (Vote Link + PEP # + Issue #) to preserve every individual exploded PEP record.

In [5]:
import json
import pandas as pd
from pymongo import MongoClient

# 1. Load MongoDB connection URI from config.json
with open("config.json") as f:
    config = json.load(f)

mongo_uri = config["mongo_uri"]

# 2. Connect to MongoDB and target 'consensus' database & 'python_decision_eps' collection
client = MongoClient(mongo_uri)
db = client["consensus"]
collection = db["python_decision_eps"]

# 3. Upsert records into MongoDB
records = df_combined.to_dict("records")

print(
    f"Upserting {len(records)} records into 'consensus.python_decision_eps'..."
)

for record in records:
    # Use composite query key so exploded PEP rows with shared Vote Links are each stored uniquely
    query_filter = {
        "Vote Link": record["Vote Link"],
        "PEP #": str(record["PEP #"]),
        "Issue #": str(record["Issue #"]),
    }
    collection.update_one(query_filter, {"$set": record}, upsert=True)

print(
    f"Successfully upserted {len(records)} documents into 'consensus.python_decision_eps'."
)

# 4. Save combined output locally as backup
excel_out = "typing_council_and_pep_submissions_combined.xlsx"
csv_out = "typing_council_and_pep_submissions_combined.csv"

df_combined.to_excel(excel_out, index=False)
df_combined.to_csv(csv_out, index=False)

print(
    f"Data preparation complete. Total expanded records: {len(df_combined)}"
)
print(f"Combined data exported to '{excel_out}' and '{csv_out}'.")

Upserting 95 records into 'consensus.python_decision_eps'...
Successfully upserted 95 documents into 'consensus.python_decision_eps'.
Data preparation complete. Total expanded records: 95
Combined data exported to 'typing_council_and_pep_submissions_combined.xlsx' and 'typing_council_and_pep_submissions_combined.csv'.
